In [1]:
import optuna
import optuna.visualization as vis

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
data = load_breast_cancer()
X, y = data.data, data.target

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# optuna

In [3]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score

In [4]:
def objective(trial):
    params = {
    'n_estimators': trial.suggest_int("n_estimators", 10, 300),
    'max_depth': trial.suggest_int("max_depth", 2, 32, log=True),
    'min_samples_split': trial.suggest_float("min_samples_split", 0.1, 1.0),
    'min_samples_leaf': trial.suggest_int("min_samples_leaf", 1, 10)
    }
    
    clf = RandomForestClassifier(**params, random_state=42, n_jobs=-1)

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    score = cross_val_score(clf, X_train, y_train, cv=cv, scoring="roc_auc")
    return score.mean()

In [5]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Найкращі параметри:", study.best_params)

[I 2026-03-09 22:45:56,721] A new study created in memory with name: no-name-618f12da-9e36-42a5-aac0-5531c03e65b8


[I 2026-03-09 22:45:57,040] Trial 0 finished with value: 0.5 and parameters: {'n_estimators': 89, 'max_depth': 2, 'min_samples_split': 0.8948838168193426, 'min_samples_leaf': 8}. Best is trial 0 with value: 0.5.
[I 2026-03-09 22:45:57,514] Trial 1 finished with value: 0.9780543395220508 and parameters: {'n_estimators': 233, 'max_depth': 14, 'min_samples_split': 0.5496527710326482, 'min_samples_leaf': 3}. Best is trial 1 with value: 0.9780543395220508.
[I 2026-03-09 22:45:57,801] Trial 2 finished with value: 0.5 and parameters: {'n_estimators': 135, 'max_depth': 2, 'min_samples_split': 0.8472749895401743, 'min_samples_leaf': 2}. Best is trial 1 with value: 0.9780543395220508.
[I 2026-03-09 22:45:58,381] Trial 3 finished with value: 0.9814067446576967 and parameters: {'n_estimators': 270, 'max_depth': 5, 'min_samples_split': 0.32565920875128096, 'min_samples_leaf': 6}. Best is trial 3 with value: 0.9814067446576967.
[I 2026-03-09 22:45:58,649] Trial 4 finished with value: 0.5 and paramet

Найкращі параметри: {'n_estimators': 176, 'max_depth': 5, 'min_samples_split': 0.1611297631709647, 'min_samples_leaf': 3}


In [6]:
best_clf = RandomForestClassifier(**study.best_params, random_state=42)
best_clf.fit(X_train, y_train)
y_pred = best_clf.predict(X_test)

print("Точність на тесті:", accuracy_score(y_test, y_pred))

Точність на тесті: 0.9649122807017544


In [6]:
vis.plot_param_importances(study)

In [7]:
vis.plot_slice(study)

In [8]:
vis.plot_contour(study, params=["min_samples_split", "min_samples_leaf"])

In [9]:
vis.plot_optimization_history(study)

# Bayes

In [ ]:
from skopt import BayesSearchCV
from skopt.space import Integer, Real

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

In [13]:
param_dist = {
    'n_estimators': Integer(10, 300),
    'max_depth': Integer(2, 32),
    'min_samples_split': Real(0.1, 1.0),
    'min_samples_leaf': Integer(1, 10)
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
model = RandomForestClassifier()

In [14]:
opt = BayesSearchCV(
    model,
    param_dist,
    n_iter=20,
    cv=cv,
    n_jobs=-1,
    scoring='roc_auc',
)

opt.fit(X_train, y_train)
model = opt.best_estimator_

In [15]:
y_pred = model.predict(X_test)
print("Точність на тесті:", accuracy_score(y_test, y_pred))

Точність на тесті: 0.9649122807017544
